In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.schema import Document
from langchain_unstructured import UnstructuredLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import pandas as pd
import os
from langchain_chroma import Chroma


def load_and_split():
  #load metadata 
  metadata_df = pd.read_csv("../resources/rwth.csv")
  # using "N" as the key
  metadata_dict = {
    row["N"]: {
        "Title" : row["Title"],
        "erschienen": row["Erschienen"],
        "nummer": row["Nummer"],
        "ordnung": row["Ordnung"],
        "version": row["Version"],
        "studiengang": row["Studiengang"],
        "abschlussart": row["AbschlussArt"]
    }
    for _, row in metadata_df.iterrows()
  }
  pdf_dir = "../resources/docs/rwth_pdfs"
  documents = []
  doc_counter = 0
  file = "1_2025-0030.pdf"
  file_number = int(file.split("_")[0])
  loader = UnstructuredLoader(
              file_path=os.path.join(pdf_dir, file),
              strategy="hi_res",
              partition_via_api=False,
              show_progress=True,
    )
  doc_elements = []
  for doc in loader.lazy_load():
    doc_metadata = metadata_dict.get(file_number)
    # Convert 'points' dictionary to string
    if 'points' in doc.metadata:
      doc.metadata['points'] = str(doc.metadata['points'])
      langchain_doc = Document(
            page_content=doc.page_content,
            metadata={**doc.metadata, **doc_metadata}
      )
      doc_elements.append(langchain_doc)
      documents.extend(doc_elements)
      doc_counter += 1
      print(f"Document {file_number} was loaded. {doc_counter}/516 files")
  print(f"{doc_counter} Documents were loaded")

In [17]:
load_and_split()

/Users/pooya/miniconda3/envs/campuswise/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: pikepdf C++ to Python logger bridge initialized
INFO: Reading PDF for file: ../resources/docs/rwth_pdfs/1_2025-0030.pdf ...


KeyboardInterrupt: 

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=2000,
      chunk_overlap=200,
      length_function=len,
      add_start_index=True,
  )
chunks = text_splitter.split_documents(documents)

